# Feature-Based Unsupervised Anomaly Detection: Fair Detector Comparison

This notebook uses the same single-record vibration dataset as `01_signal_analysis_walkthrough_v2.ipynb`, but focuses on a narrower question: do several interpretable unsupervised scoring strategies prioritize the same candidate region when they receive the same features?

The dataset is unlabeled during modeling. Model outputs are inspection rankings, not confirmed fault labels. The injected synthetic event is revealed only after comparison and sensitivity analysis are complete.

## 1. Setup And Standalone Context

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = PROJECT_ROOT / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from signal_processing_prep.artifacts import FeatureTable
from signal_processing_prep.data_loading import SignalDatasetLoader
from signal_processing_prep.features import FeatureExtractor, FrequencyBand, SlidingWindowConfig
from signal_processing_prep.modeling import (
    DbscanOutlierScorer,
    IsolationForestScorer,
    LocalOutlierFactorScorer,
    OneClassSvmScorer,
    PcaReconstructionScorer,
    RobustMahalanobisScorer,
    RobustZScoreScorer,
    consensus_candidate_regions,
    evaluate_candidate_region_overlap,
    group_candidate_regions,
)
from signal_processing_prep.plotting import plot_frequency_spectrum, plot_spectrogram_dynamic_range, plot_time_signal, plot_time_signal_adaptive, plot_wavelet_scalogram
from signal_processing_prep.quality import assess_signal_quality

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "vibration_anomaly_single_record.csv"

In [ ]:
from IPython import get_ipython

ip = get_ipython()
if ip is not None:
    try:
        ip.run_line_magic("matplotlib", "widget")
        print("Using matplotlib widget backend.")
    except Exception:
        ip.run_line_magic("matplotlib", "inline")
        print("Widget backend unavailable; using inline plots.")

In [ ]:
record = SignalDatasetLoader().load_file(DATA_PATH, signal_column="voltage")
quality = assess_signal_quality(record)
print(f"Samples: {record.n_samples:,}; duration: {record.duration_seconds:.2f} s; sampling rate: {record.sampling_rate_hz:.1f} Hz")
display(pd.DataFrame([quality.to_dict()]))

fig, ax = plot_time_signal_adaptive(record, max_points=2000)
ax.set_ylabel("Voltage [V]")
ax.set_title("Record overview before feature scoring")
fig, ax = plot_frequency_spectrum(record, spectrum_type="psd", nperseg=4 * 4096)
ax.set_yscale("log")
ax.set_title("Full-record PSD context")
fig, ax = plot_spectrogram_dynamic_range(record, window_seconds=0.1, step_seconds=0.05, max_frequency_hz=6000.0, frequency_scale="log")
ax.set_title("Full-record spectrogram context")

The context plots are intentionally brief here. Notebook 01 explains the investigation path; this notebook concentrates on feature selection, detector assumptions, and stability of ranked regions.

## 2. Build One Shared Feature Table

Five physically interpretable features are inspected below. For the primary detector comparison, every detector receives the same validated two-feature input (`rms` and `kurtosis`): this retains amplitude and impulsiveness information while avoiding a rank-deficient robust covariance fit caused by strongly coupled energy features.

In [ ]:
window_config = SlidingWindowConfig(
    window_seconds=0.20,
    step_seconds=0.025,
    frequency_bands=(
        FrequencyBand("rotating_40_500", 40.0, 500.0),
        FrequencyBand("resonance_1800_3200", 1800.0, 3200.0),
        FrequencyBand("broadband_3200_6000", 3200.0, 6000.0),
    ),
    frequency_window="hann",
    normalize_frequency_window_power=True,
)
full_feature_table = FeatureExtractor().extract_windows(record, window_config)
features = full_feature_table.to_dataframe()

inspection_feature_columns = (
    "rms",
    "crest_factor",
    "kurtosis",
    "band_energy_resonance_1800_3200",
    "band_energy_broadband_3200_6000",
)
selected_feature_columns = ("rms", "kurtosis")
context_columns = [column for column in ["record_name", "source_name", "label", "window_start_seconds", "window_end_seconds", "window_center_seconds", "window_n_samples", "frequency_window"] if column in features.columns]
inspection_frame = features.loc[:, context_columns + list(inspection_feature_columns)].copy()
comparison_frame = features.loc[:, context_columns + list(selected_feature_columns)].copy()
comparison_feature_table = FeatureTable.from_dataframe(comparison_frame, feature_columns=selected_feature_columns)

print(f"All extracted feature columns: {len(features.columns)}")
print(f"Inspected evidence features: {list(inspection_feature_columns)}")
print(f"Validated shared model inputs: {list(selected_feature_columns)}")
inspection_frame.head()

## 3. Inspect Features Before Fitting Models

The inspected variables express amplitude, impulsiveness, and energy in two hypothesized higher-frequency bands. Feature correlation is checked before fitting models because redundant measurements can destabilize covariance- and distance-based methods.

In [ ]:
fig, axes = plt.subplots(len(inspection_feature_columns), 1, figsize=(11, 11), sharex=True)
for ax, feature in zip(axes, inspection_feature_columns):
    ax.plot(inspection_frame["window_center_seconds"], inspection_frame[feature], linewidth=1.0)
    ax.set_ylabel(feature.replace("band_energy_", "energy\n"), fontsize=8)
    ax.grid(True, alpha=0.25)
axes[0].set_title("Selected interpretable feature timelines")
axes[-1].set_xlabel("Window center [s]")
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, len(inspection_feature_columns), figsize=(16, 3.5))
for ax, feature in zip(axes, inspection_feature_columns):
    ax.hist(inspection_frame[feature], bins=20, alpha=0.8)
    ax.set_title(feature.replace("band_energy_", "energy\n"), fontsize=8)
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel("Window count")
fig.suptitle("Feature distributions before anomaly scoring", y=1.04)
fig.tight_layout()

correlation = inspection_frame.loc[:, inspection_feature_columns].corr(method="spearman")
fig, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(correlation, cmap="coolwarm", vmin=-1.0, vmax=1.0)
ax.set_xticks(range(len(inspection_feature_columns)), labels=inspection_feature_columns, rotation=45, ha="right")
ax.set_yticks(range(len(inspection_feature_columns)), labels=inspection_feature_columns)
for row in range(len(inspection_feature_columns)):
    for column in range(len(inspection_feature_columns)):
        ax.text(column, row, f"{correlation.iloc[row, column]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Spearman correlation of selected features")
fig.colorbar(image, ax=ax, label="Spearman correlation")
fig.tight_layout()
correlation

**Interpretation:** high-band energies remain useful evidence for later physical interpretation, but they are strongly coupled with amplitude behavior in this record. The primary model comparison therefore uses the declared non-degenerate `rms` and `kurtosis` subset for every detector; higher-band evidence is not allowed to make only one detector numerically unstable.

## 4. Detector Assumptions And Default Settings

In [ ]:
detector_assumptions = pd.DataFrame(
    [
        ("Robust positive z-score", "sum of unusually high feature deviations", "transparent one-sided changes", "selected features", "misses unusual decreases"),
        ("Robust Mahalanobis", "robust multivariate distance", "joint deviations", "support_fraction=None", "sensitive to degeneracy/correlation"),
        ("Isolation Forest", "ease of isolating a row", "nonlinear multifeature outliers", "contamination=0.03", "ranking depends on feature representation"),
        ("One-class SVM", "distance outside learned normal-like boundary", "compact nonlinear boundary", "nu=0.03", "parameter-sensitive on small sets"),
        ("Local Outlier Factor", "local neighborhood sparsity", "local departures", "n_neighbors=10", "overlapping windows distort neighborhoods"),
        ("PCA reconstruction", "reconstruction error", "off-pattern variation", "n_components=0.95", "normal structure not independently trained"),
        ("DBSCAN diagnostic", "sparse/noise cluster behavior", "exploratory clustering check", "eps=1.5", "eps-sensitive; not primary evidence"),
    ],
    columns=["Detector", "Score meaning", "Useful behavior", "Default parameter", "Important failure mode"],
)
detector_assumptions

## 5. Primary Comparison On Identical Inputs

In [ ]:
evaluations = {
    "robust_positive_z": RobustZScoreScorer(feature_columns=selected_feature_columns).score(comparison_feature_table),
    "robust_mahalanobis": RobustMahalanobisScorer(feature_columns=selected_feature_columns, random_state=0).score(comparison_feature_table),
    "isolation_forest": IsolationForestScorer(contamination=0.03, random_state=0).score(comparison_feature_table),
    "one_class_svm": OneClassSvmScorer(nu=0.03).score(comparison_feature_table),
    "local_outlier_factor": LocalOutlierFactorScorer(n_neighbors=10, contamination=0.03).score(comparison_feature_table),
    "pca_reconstruction": PcaReconstructionScorer(n_components=0.95).score(comparison_feature_table),
    "dbscan_diagnostic": DbscanOutlierScorer(eps=1.5, min_samples=8).score(comparison_feature_table),
}
primary_top_regions = pd.concat(
    [group_candidate_regions(result, top_n=20, merge_gap_seconds=window_config.step_seconds).head(3).assign(method=method) for method, result in evaluations.items()],
    ignore_index=True,
)
primary_top_regions[["method", "region_rank", "region_start_seconds", "region_end_seconds", "window_count", "best_window_rank"]]

In [ ]:
def minmax_score(values):
    values = pd.to_numeric(values, errors="coerce")
    spread = values.max() - values.min()
    return pd.Series(np.zeros(len(values)), index=values.index) if spread == 0 else (values - values.min()) / spread

score_frame = comparison_frame[["window_start_seconds", "window_end_seconds", "window_center_seconds"]].copy()
for method, evaluation in evaluations.items():
    aligned = evaluation.prediction_frame.sort_values("row_index")
    score_frame[method] = minmax_score(aligned["anomaly_score"]).to_numpy()

fig, ax = plt.subplots(figsize=(12, 5))
for method in evaluations:
    ax.plot(score_frame["window_center_seconds"], score_frame[method], linewidth=1.0, alpha=0.75, label=method)
ax.set_title("Normalized anomaly scores using one shared feature table")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Min-max normalized score")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right", ncols=2)
fig.tight_layout()

score_rank_correlation = score_frame[list(evaluations)].corr(method="spearman")
score_rank_correlation

## 6. Agreement At Candidate-Region Level

Window-level agreement exaggerates evidence because windows overlap strongly. Candidate windows are grouped into event-like regions first. DBSCAN is shown above as a diagnostic, but excluded from the primary consensus because its clustering result is especially dependent on `eps`.

In [ ]:
primary_evaluations = {method: result for method, result in evaluations.items() if method != "dbscan_diagnostic"}
consensus_regions = consensus_candidate_regions(primary_evaluations, top_n_per_method=20, merge_gap_seconds=window_config.step_seconds)
consensus_regions.head(10)

**Interpretation:** detector agreement demonstrates stability across scoring assumptions applied to the same features. It is not independent confirmation of a fault because every detector sees the same overlapping windows from one acquisition.

In [ ]:
top_consensus = consensus_regions.iloc[0]
candidate_center = 0.5 * (float(top_consensus["region_start_seconds"]) + float(top_consensus["region_end_seconds"]))
zoom_start = max(candidate_center - 0.6, 0.0)
zoom_duration = min(1.2, record.duration_seconds - zoom_start)
fig, ax = plot_time_signal(record, start_seconds=zoom_start, duration_seconds=zoom_duration, max_points=8000)
ax.axvspan(top_consensus["region_start_seconds"], top_consensus["region_end_seconds"], color="tab:red", alpha=0.2, label="top consensus region")
ax.set_ylabel("Voltage [V]")
ax.set_title("Raw signal around top consensus candidate")
ax.legend(loc="best")

start_index = int(round(zoom_start * record.sampling_rate_hz))
end_index = min(int(round((zoom_start + zoom_duration) * record.sampling_rate_hz)), record.n_samples)
zoom_record = record.segment(start_index, end_index, index=0)
fig, ax = plot_spectrogram_dynamic_range(zoom_record, window_seconds=0.05, step_seconds=0.0025, max_frequency_hz=6000.0, frequency_scale="log")
ax.set_title("Top-consensus area spectrogram; relative time")
fig, ax = plot_wavelet_scalogram(zoom_record, min_frequency_hz=20.0, max_frequency_hz=6000.0, n_frequencies=128, frequency_scale="log")
ax.set_title("Top-consensus area wavelet scalogram; relative time")

## 7. Parameter Sensitivity Without Consulting The Synthetic Answer Key

Each variant uses the same selected feature table. The question is whether its top grouped region continues to overlap the default consensus region, not whether a chosen parameter maximizes agreement with hidden truth.

In [ ]:
variant_scorers = {
    "mahalanobis_auto": RobustMahalanobisScorer(feature_columns=selected_feature_columns, support_fraction=None, random_state=0),
    "mahalanobis_0p75": RobustMahalanobisScorer(feature_columns=selected_feature_columns, support_fraction=0.75, random_state=0),
    "iforest_0p01": IsolationForestScorer(contamination=0.01, random_state=0),
    "iforest_0p03": IsolationForestScorer(contamination=0.03, random_state=0),
    "iforest_0p05": IsolationForestScorer(contamination=0.05, random_state=0),
    "ocsvm_0p01": OneClassSvmScorer(nu=0.01),
    "ocsvm_0p03": OneClassSvmScorer(nu=0.03),
    "ocsvm_0p05": OneClassSvmScorer(nu=0.05),
    "lof_5": LocalOutlierFactorScorer(n_neighbors=5, contamination=0.03),
    "lof_10": LocalOutlierFactorScorer(n_neighbors=10, contamination=0.03),
    "lof_20": LocalOutlierFactorScorer(n_neighbors=20, contamination=0.03),
    "pca_0p80": PcaReconstructionScorer(n_components=0.80),
    "pca_0p90": PcaReconstructionScorer(n_components=0.90),
    "pca_0p95": PcaReconstructionScorer(n_components=0.95),
    "dbscan_1p0": DbscanOutlierScorer(eps=1.0, min_samples=8),
    "dbscan_1p5": DbscanOutlierScorer(eps=1.5, min_samples=8),
    "dbscan_2p0": DbscanOutlierScorer(eps=2.0, min_samples=8),
}
default_start = float(top_consensus["region_start_seconds"])
default_end = float(top_consensus["region_end_seconds"])
sensitivity_rows = []
for variant, scorer in variant_scorers.items():
    result = scorer.score(comparison_feature_table)
    top_region = group_candidate_regions(result, top_n=20, merge_gap_seconds=window_config.step_seconds).iloc[0]
    overlap = max(0.0, min(float(top_region["region_end_seconds"]), default_end) - max(float(top_region["region_start_seconds"]), default_start))
    sensitivity_rows.append(
        {
            "variant": variant,
            "top_region_start_seconds": top_region["region_start_seconds"],
            "top_region_end_seconds": top_region["region_end_seconds"],
            "overlap_with_default_consensus_seconds": overlap,
            "overlaps_default_consensus": overlap > 0.0,
        }
    )
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity

**Reading the table:** agreement across reasonable parameters strengthens the decision to inspect a time region. Sensitivity, especially for DBSCAN, is itself a useful warning against reporting a single score as a calibrated result.

## 8. Feature-Set Sensitivity Appendix

The primary conclusion uses interpretable declared features. As an appendix, compare it with models that accept all calculated numeric features produced by the extractor.

In [ ]:
broad_evaluations = {
    "robust_positive_z": RobustZScoreScorer().score(full_feature_table),
    "isolation_forest": IsolationForestScorer(contamination=0.03, random_state=0).score(full_feature_table),
    "one_class_svm": OneClassSvmScorer(nu=0.03).score(full_feature_table),
    "local_outlier_factor": LocalOutlierFactorScorer(n_neighbors=10, contamination=0.03).score(full_feature_table),
    "pca_reconstruction": PcaReconstructionScorer(n_components=0.95).score(full_feature_table),
}
broad_consensus = consensus_candidate_regions(broad_evaluations, top_n_per_method=20, merge_gap_seconds=window_config.step_seconds)
broad_top = broad_consensus.iloc[0]
broad_overlap = max(0.0, min(float(broad_top["region_end_seconds"]), default_end) - max(float(broad_top["region_start_seconds"]), default_start))
pd.DataFrame(
    [
        {"feature_policy": "selected interpretable features", "top_start_seconds": default_start, "top_end_seconds": default_end, "method_count": top_consensus["method_count"]},
        {"feature_policy": "all calculated numeric features", "top_start_seconds": broad_top["region_start_seconds"], "top_end_seconds": broad_top["region_end_seconds"], "method_count": broad_top["method_count"]},
    ]
).assign(overlaps_primary_top_region=[True, broad_overlap > 0.0])

## 9. Truth Reveal For Synthetic Validation Only

Only now is the injected synthetic interval made available. This final check assesses whether the default consensus localized the designed event; it was not used for feature selection, model fitting, parameter sensitivity, or region agreement.

In [ ]:
from signal_processing_prep.demo_data import VIBRATION_ANOMALY_EVENT_DURATION_SECONDS, VIBRATION_ANOMALY_START_SECONDS
from signal_processing_prep.records import AnnotationInterval

withheld_interval = AnnotationInterval(
    VIBRATION_ANOMALY_START_SECONDS,
    VIBRATION_ANOMALY_START_SECONDS + VIBRATION_ANOMALY_EVENT_DURATION_SECONDS,
    kind="injected_synthetic_event",
)
validated_consensus = evaluate_candidate_region_overlap(consensus_regions, [withheld_interval])
print(f"Withheld injected interval: {withheld_interval.start_seconds:.3f} s to {withheld_interval.end_seconds:.3f} s")
validated_consensus.head(10)

## 10. Practical Conclusion

- A robust positive z-score is preferred when the engineering question is explicitly about unusually high, physically interpretable quantities and transparent explanation matters most.
- More flexible unsupervised models are useful as sensitivity checks when multivariate or local structure may matter, but they introduce parameter and representation dependence.
- Consensus across these models indicates a stable inspection priority, not confirmed failure evidence.
- A real validation program requires repeated independent records, operating-condition context, and confirmed outcomes before thresholds or model performance can be claimed.
- Neural reconstruction methods remain a separate controlled experiment in `04_synthetic_spectrogram_autoencoder_walkthrough.ipynb`, not evidence for this one unlabeled run.